In [1]:
import pickle
import os
import pandas as pd

In [2]:
os.getcwd()

'f:\\Project\\Spam Classifer\\notebook'

In [3]:
df = pd.read_csv('../data/processed/cleaned_data.csv')
df.head()

,category,Message
0,ham,go jurong pointsymtoken crazysymtokensymtoken ...
1,ham,ok larsymtokensymtokensymtoken joke wif u onis...
2,spam,free entri numtoken wkli comp win fa cup final...
3,ham,u dun say earli horsymtokensymtokensymtoken u ...
4,ham,nah donsymtok think goe usfsymtoken live aroun...


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['category'] = le.fit_transform(df['category'])

tfid = TfidfVectorizer(max_features=3000)
X = tfid.fit_transform(df['Message']).toarray()
y = df['category'].values

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
X_train.shape

(4135, 3000)

In [10]:
y_train.shape

(4135,)

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [12]:
models = {
    'Logistic Regression': LogisticRegression(),
    'MultinomialNB' : MultinomialNB(),
    'BernoulliNB' : BernoulliNB(),
    'SVM' : SVC(kernel='linear'),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
    'Gradient Boosting': GradientBoostingClassifier()
}

result = []



In [14]:

for name , model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    result.append({
        'Model': name,
        'Accuracy' : accuracy_score(y_test, y_pred),
        'Precision' : precision_score(y_test, y_pred),
        'Recall' : recall_score(y_test, y_pred),
        'f1 Score' : f1_score(y_test, y_pred)
    })
    

In [16]:
result_df = pd.DataFrame(result).sort_values('f1 Score', ascending= False)

In [17]:
result_df

,Model,Accuracy,Precision,Recall,f1 Score
2,BernoulliNB,0.978723,0.984252,0.862069,0.919118
5,Random Forest,0.978723,1.000000,0.848276,0.917910
3,SVM,0.976789,0.954887,0.875862,0.913669
6,Gradient Boosting,0.976789,0.984000,0.848276,0.911111
1,MultinomialNB,0.974855,1.000000,0.820690,0.901515
4,Decision Tree,0.968085,0.878378,0.896552,0.887372
0,Logistic Regression,0.957447,0.939130,0.744828,0.830769


In [18]:
for alpha in [0.1, 0.3, 0.5, 0.7, 1.0]:
    model = BernoulliNB(alpha=alpha)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    print(f"Alpha: {alpha} | Precision: {precision:.4f} | F1: {f1:.4f}")

Alpha: 0.1 | Precision: 0.9773 | F1: 0.9314
Alpha: 0.3 | Precision: 0.9769 | F1: 0.9236
Alpha: 0.5 | Precision: 0.9769 | F1: 0.9236
Alpha: 0.7 | Precision: 0.9767 | F1: 0.9197
Alpha: 1.0 | Precision: 0.9843 | F1: 0.9191


In [19]:
from sklearn.metrics import classification_report

In [20]:
final_model = BernoulliNB(alpha= 0.1)
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       889
           1       0.98      0.89      0.93       145

    accuracy                           0.98      1034
   macro avg       0.98      0.94      0.96      1034
weighted avg       0.98      0.98      0.98      1034



In [21]:
os.makedirs('../model', exist_ok=True)

In [23]:
with open('../model/spam_classifier.pkl', 'wb') as f:
    pickle.dump(final_model, f)